<a href="https://colab.research.google.com/github/brahime2000-hue/contract-app/blob/main/Builder.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [111]:
# @title
# Prompt Builder — Luxe Purple UI (tab-to-copy, template prompts)
import sys, subprocess, json, re
import ipywidgets as widgets
from IPython.display import display, HTML, Javascript

# Quiet install
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "yt-dlp"], check=True)
import yt_dlp  # noqa: E402

PROMPT_COUNT = 5
prompt_data = [""] * PROMPT_COUNT
last_title = None

# ---------------- Styles ----------------
display(HTML(r"""
<style>
/* Inputs & dropdowns */
.widget-text input,
.widget-dropdown select {
  background: #120f22 !important;
  border: 1px solid rgba(178,119,255,0.28) !important;
  color: #f5f3ff !important;
  border-radius: 14px !important;
  padding: 12px 14px !important;
  height: 44px !important;
  font-size: 14px !important;
  font-weight: 700 !important;
  box-shadow: 0 6px 14px rgba(0,0,0,0.35);
  transition: border-color 0.2s, box-shadow 0.2s;
}
.widget-text input::placeholder,
.widget-dropdown select option { color: rgba(245,243,255,0.75); }
.widget-text input:focus,
.widget-dropdown select:focus {
  border-color: var(--accent) !important;
  box-shadow: 0 0 0 3px rgba(178,119,255,0.16);
}
.widget-dropdown select {
  appearance: none;
  background-image:
    linear-gradient(45deg, transparent 50%, #d9a7ff 50%),
    linear-gradient(135deg, #d9a7ff 50%, transparent 50%),
    linear-gradient(to right, transparent, transparent);
  background-position:
    calc(100% - 18px) calc(50% - 3px),
    calc(100% - 12px) calc(50% - 3px),
    calc(100% - 40px) 50%;
  background-size: 6px 6px, 6px 6px, 1px 40px;
  background-repeat: no-repeat;
}

@import url('https://fonts.googleapis.com/css2?family=Inter:wght@400;500;600;700;800&display=swap');
:root { --bg:#0d0a14; --panel:#18142c; --panel2:#1f1a35; --accent:#b277ff; --accent2:#d9a7ff; --text:#f5f3ff; --muted:#b6a8d9; --radius:20px; }
body, .jp-Notebook, .jp-Cell, .jp-OutputArea-output {
  background: radial-gradient(120% 90% at 20% 0%, #1f1534 0%, #0d0a14 45%, #0a0811 100%) !important;
  color: var(--text) !important; font-family:'Inter',system-ui,sans-serif !important;
}
.section-card { background: linear-gradient(145deg,#18142c,#1f1a35); border:1px solid rgba(178,119,255,0.25); border-radius:var(--radius);
  padding:22px; box-shadow:0 18px 50px -14px rgba(0,0,0,0.65); margin-bottom:18px; }
.hero-wrap { background: linear-gradient(135deg,#1c1531,#120f22); border:1px solid rgba(178,119,255,0.35);
  border-radius:22px; padding:28px 32px; margin-bottom:20px; box-shadow:0 24px 60px -18px rgba(0,0,0,0.65); }
.brand-row { display:flex; align-items:center; gap:12px; }
.brand-dot { width:22px; height:22px; border-radius:9999px; background: radial-gradient(circle at 30% 30%, #fff, #b277ff 55%, #6a34ff 100%);
  box-shadow:0 0 24px rgba(178,119,255,0.65); }
.hero-title { font-size:38px; font-weight:800; letter-spacing:-1.4px; margin:8px 0 0 0; line-height:1.05; }
.hero-title span { color: var(--accent); }

.widget-textarea textarea { background:#120f22 !important; border:1px solid rgba(178,119,255,0.22) !important; border-radius:16px !important;
  padding:16px 18px !important; font-size:14.5px !important; color:var(--text) !important; font-family:ui-monospace, monospace !important; min-height:240px !important; }
.widget-textarea textarea:focus { border-color: var(--accent) !important; box-shadow:0 0 0 3px rgba(178,119,255,0.18) !important; }

.segmented-control {
  background: linear-gradient(145deg,#1a162d,#110d20); border:1px solid rgba(178,119,255,0.28);
  border-radius:9999px; padding:6px 8px; display:flex; flex-wrap:wrap; gap:8px; width:100%;
  box-shadow: inset 0 2px 5px rgba(0,0,0,0.35);
}
.segmented-tab {
  flex: 0 0 auto; padding:10px 16px; text-align:center; font-weight:700; font-size:13.5px;
  border-radius:9999px; cursor:pointer; color:var(--muted); transition: all 0.25s cubic-bezier(0.4,0,0.2,1);
  background: transparent; border: none;
}
.segmented-tab:hover { color:#e5ddff; background: rgba(255,255,255,0.04); }
.segmented-tab.active {
  background: linear-gradient(135deg, var(--accent), var(--accent2)); color:#130c21;
  box-shadow: 0 6px 16px rgba(178,119,255,0.35), inset 0 1px 0 rgba(255,255,255,0.45);
  transform: translateY(-1px);
}

.prompt-output {
  background:#120f22; border:1px solid rgba(178,119,255,0.20); border-radius:16px; padding:20px; min-height:400px;
  font-family: ui-monospace, monospace; font-size:14.2px; line-height:1.65; white-space: pre-wrap; color:#e9e5ff;
}

.status-text {
  background: rgba(178,119,255,0.10); border:1px solid rgba(178,119,255,0.28); color: var(--accent2);
  padding:10px 18px; border-radius:14px; font-size:13.5px; font-weight:600;
}

/* Generate button (teal) */
.generate-btn {
  background: linear-gradient(135deg, #6ff0e5, #7bc5ff);
  position: relative; color: #0c0f16; font-weight:800; font-size:15px; padding:14px 30px;
  border-radius:18px; border:none; box-shadow:0 15px 32px rgba(111,240,229,0.28), inset 0 1px 0 rgba(255,255,255,0.32);
  cursor:pointer; height:54px;
}
.generate-btn:after {
  content:""; position:absolute; inset:-2px; border-radius:20px;
  background: radial-gradient(120% 80% at 30% 20%, rgba(255,255,255,0.28), transparent 55%);
  pointer-events:none;
}
.generate-btn:hover { transform: translateY(-1.5px); box-shadow:0 18px 34px rgba(111,240,229,0.4); }

/* Red state when no link (or empty) */
.generate-btn-danger {
  background: linear-gradient(135deg, #ff5f6d, #e5383b);
  position: relative; color: #fff; font-weight:800; font-size:15px; padding:14px 30px;
  border-radius:18px; border:none; box-shadow:0 15px 32px rgba(229,56,59,0.30), inset 0 1px 0 rgba(255,255,255,0.24);
  cursor:pointer; height:54px;
}
.generate-btn-danger:after {
  content:""; position:absolute; inset:-2px; border-radius:20px;
  background: radial-gradient(120% 80% at 30% 20%, rgba(255,255,255,0.22), transparent 55%);
  pointer-events:none;
}
.generate-btn-danger:hover { transform: translateY(-1px); box-shadow:0 18px 34px rgba(229,56,59,0.40); }

/* Clear button: same palette as generate, smaller */
.clear-btn {
  background: linear-gradient(135deg, #6ff0e5, #7bc5ff);
  color: #0c0f16;
  font-weight: 800;
  font-size: 13.5px;
  letter-spacing: 0.2px;
  padding: 11px 18px;
  border-radius: 14px;
  border: 1px solid rgba(255,255,255,0.18);
  box-shadow:
    0 12px 24px rgba(0,0,0,0.30),
    inset 0 1px 0 rgba(255,255,255,0.32);
  cursor: pointer;
  height: 46px;
  text-shadow: 0 1px 2px rgba(255,255,255,0.35), 0 1px 3px rgba(0,0,0,0.22);
}
.clear-btn:hover {
  transform: translateY(-1px);
  box-shadow:
    0 14px 28px rgba(0,0,0,0.34),
    inset 0 1px 0 rgba(255,255,255,0.38);
}

.control-row { display:flex; align-items:center; justify-content:flex-start; gap:12px; margin:10px 0 8px 0; flex-wrap:wrap; }
</style>
"""))

# ---------------- Helpers ----------------
def extract_youtube_url(text: str | None):
    if not text: return None
    match = re.search(r"https?://\S+", text)
    if not match: return None
    url = match.group(0).strip()
    return url if any(x in url for x in ["youtube.com/watch", "youtu.be/", "youtube.com/shorts"]) else None

def get_video_title_from_url(url: str) -> str | None:
    ydl_opts = {"quiet": True, "no_warnings": True}
    try:
        with yt_dlp.YoutubeDL(ydl_opts) as ydl:
            info = ydl.extract_info(url, download=False)
        return (info.get("title") or "").strip()
    except Exception:
        return None

def clean_reference_script(text: str | None) -> str:
    if not text:
        return ""
    text = re.sub(r"https?://\S+", "", text)
    text = re.sub(r"(?im)^transcript\s*:?.*$", "", text)
    cleaned = []
    for line in text.splitlines():
        line = line.strip()
        if not line or line == ":":
            continue
        cleaned.append(line)
    return "\n".join(cleaned)

# ---------------- Template Prompts ----------------
PROMPT_1 = """
REFERENCE SCRIPT 1
[REFERENCE_SCRIPT_START]
[TEST1FORPROMPT1]
[REFERENCE_SCRIPT_END]
"""
PROMPT_2 = """
INPUTS
New video title: [TEST1FORPROMPT2]
Target total word count: [TEST1FORPROMPT3]
"""
PROMPT_3 = """
LANGUAGE
- Input: [TARGET_LANGUAGE] = [TEST2FORPROMPT3]
- If missing/empty/undefined/"auto": set [TARGET_LANGUAGE]="English".
- If explicitly set: write ALL story text (hooks/sections) in exactly [TARGET_LANGUAGE].
- Never ask the user about language.




"""
PROMPT_4 = """RESPONSE STYLE
- Diagnostic: short paragraphs +TEST1FORPROMPT4
- Rewrite: ONLY narrative script text + fixed English labels/questions (no bullets/headings/meta).
- Post-analysis: short paragraphs + bullets.




"""
PROMPT_5 = """
INPUT CONTROL
TARGET_LANGUAGE: [TEST1FORPROMPT5]
- If empty or "auto" → use the language of the script
- If specified → ALL outputs MUST be in this language
"""

# ---------------- Widgets ----------------
reference_input = widgets.Textarea(
    placeholder="Paste reference script here...\n(YouTube link on first line will be auto-detected)",
    layout=widgets.Layout(width="100%", height="240px")
)
target_words = widgets.Text(value="3000", layout=widgets.Layout(width="180px"))
target_language = widgets.Dropdown(
    options=["English", "Arabic", "French", "German", "Russian", "Italian"],
    value="English",
    layout=widgets.Layout(width="200px")
)
status_bar = widgets.HTML(value="<div id='statusBar' class='status-text'>Ready — paste script and generate</div>")
prompt_areas = [widgets.Textarea(layout=widgets.Layout(visibility="hidden", height="0px")) for _ in range(PROMPT_COUNT)]

segmented_control = widgets.HTML("""
<div class="segmented-control" id="segmentedTabs">
  <button class="segmented-tab active" data-index="0">Prompt 1</button>
  <button class="segmented-tab" data-index="1">Prompt 2</button>
  <button class="segmented-tab" data-index="2">Prompt 3</button>
  <button class="segmented-tab" data-index="3">Prompt 4</button>
  <button class="segmented-tab" data-index="4">Prompt 5</button>
</div>
<div id="promptOutput" class="prompt-output">Generate prompts to see content here...</div>
<script>
window.promptData = Array(5).fill("Generate prompts to see content here...");
function switchPrompt(index) {
  const tabs = document.querySelectorAll('.segmented-tab');
  tabs.forEach(tab => tab.classList.remove('active'));
  tabs[index].classList.add('active');
  const output = document.getElementById('promptOutput');
  output.textContent = window.promptData[index] || "Generate prompts to see content here...";
}
async function copyPrompt(index) {
  const text = window.promptData[index] || "";
  if (!text.trim()) return;
  try {
    await navigator.clipboard.writeText(text);
    const status = document.getElementById('statusBar');
    if (status) status.innerHTML = `✓ Prompt ${index + 1} copied`;
    const clearBtn = document.querySelector('button.clear-btn');
    if (clearBtn && index === 4) clearBtn.style.display = 'inline-flex';
  } catch (e) {
    const status = document.getElementById('statusBar');
    if (status) status.innerHTML = `✗ Copy failed`;
  }
}
document.querySelectorAll('.segmented-tab').forEach(tab => {
  tab.addEventListener('click', () => {
    const idx = parseInt(tab.dataset.index);
    if (tab.classList.contains('active')) {
      copyPrompt(idx);          // second click copies
    } else {
      switchPrompt(idx);        // first click selects
    }
  });
});
window.switchPrompt = switchPrompt;
window.copyPrompt = copyPrompt;
</script>
""")

# ---------------- Prompt builder ----------------
def fill_template(tmpl: str, replacements: dict) -> str:
    out = tmpl
    for key, val in replacements.items():
        out = out.replace(f"[{key}]", val)
    return out

def build_prompts(title: str | None):
    global prompt_data
    ref = clean_reference_script(reference_input.value)
    words = target_words.value.strip() or "3000"
    lang = target_language.value.strip() or "English"
    vid_title = title or "Auto-detected title"

    repl = {
        "TEST1FORPROMPT1": ref,
        "TEST1FORPROMPT2": vid_title,
        "TEST1FORPROMPT3": words,
        "TEST2FORPROMPT3": lang,
        "TEST1FORPROMPT4": lang,
        "TEST1FORPROMPT5": lang,
    }

    prompt_data = [
        fill_template(PROMPT_1, repl).strip(),
        fill_template(PROMPT_2, repl).strip(),
        fill_template(PROMPT_3, repl).strip(),
        fill_template(PROMPT_4, repl).strip(),
        fill_template(PROMPT_5, repl).strip(),
    ]
    for i, area in enumerate(prompt_areas):
        area.value = prompt_data[i]

def push_prompts_to_js():
    display(Javascript(f"""
        window.promptData = {json.dumps(prompt_data)};
        if (window.switchPrompt) window.switchPrompt(0);
    """))

def update_outputs(title: str | None):
    build_prompts(title)
    push_prompts_to_js()

# ---------------- Buttons ----------------
generate_btn = widgets.Button(
    description="✦  GENERATE PROMPTS",
    layout=widgets.Layout(width="auto", height="54px")
)
generate_btn.add_class("generate-btn")  # start teal

clear_btn = widgets.Button(
    description="⟳ CLEAR ALL",
    layout=widgets.Layout(width="auto", height="46px", display="none")
)
clear_btn.add_class("clear-btn")

def update_generate_state(change=None):
    # Teal when any text present; red only when empty
    has_text = bool(reference_input.value.strip())
    generate_btn.remove_class("generate-btn")
    generate_btn.remove_class("generate-btn-danger")
    if has_text:
        generate_btn.add_class("generate-btn")
    else:
        generate_btn.add_class("generate-btn-danger")

def on_generate(_):
    global last_title
    url = extract_youtube_url(reference_input.value)
    if not url:
        status_bar.value = "<div id='statusBar' class='status-text'>✗ Please paste a YouTube link first</div>"
        update_generate_state()
        return
    status_bar.value = "<div id='statusBar' class='status-text'>Generating prompts...</div>"
    title = get_video_title_from_url(url)
    last_title = title
    update_outputs(title)
    status_bar.value = "<div id='statusBar' class='status-text'>✓ Prompts ready</div>"
    update_generate_state()

def on_clear(_):
    reference_input.value = ""
    # Keep WORDS and LANGUAGE as user set; only clear script/prompts
    reset_prompts()
    clear_btn.layout.display = "none"
    status_bar.value = "<div id='statusBar' class='status-text'>Cleared — paste script and generate</div>"
    update_generate_state()

generate_btn.on_click(on_generate)
clear_btn.on_click(on_clear)
reference_input.observe(update_generate_state, names="value")
update_generate_state()

control_row = widgets.HBox(
    [generate_btn, clear_btn],
    layout=widgets.Layout(justify_content="flex-start", align_items="center", width="100%")
)

# ---------------- Layout ----------------
display(HTML("""
<div class="hero-wrap">
  <div class="brand-row">
    <div class="brand-dot"></div>
    <div style="font-weight:800; font-size:22px;">Prompt Builder</div>
  </div>
  <div class="hero-title">Modern <span>Kamilys</span></div>
</div>
"""))

display(HTML("<div class='section-card'></div>"))
display(widgets.HTML("<div style='font-size:11px; letter-spacing:1.5px; text-transform:uppercase; font-weight:700; color:#b6a8d9; margin-bottom:10px;'>REFERENCE SCRIPT</div>"))
display(reference_input)
display(HTML("<div></div>"))

display(HTML("<div class='section-card'></div>"))
display(widgets.HTML("<div style='font-size:11px; letter-spacing:1.5px; text-transform:uppercase; font-weight:700; color:#b6a8d9; margin-bottom:12px;'>SETTINGS</div>"))
display(widgets.HBox([
    widgets.VBox([widgets.HTML("<div style='font-size:11px;color:#b6a8d9;margin-bottom:6px;'>WORDS</div>"), target_words]),
    widgets.VBox([widgets.HTML("<div style='font-size:11px;color:#b6a8d9;margin-bottom:6px;'>LANGUAGE</div>"), target_language]),
], layout=widgets.Layout(gap="24px")))
display(HTML("<div style='height:10px'></div>"))
display(control_row)
display(widgets.HTML("<div style='height:8px'></div>"))
display(status_bar)
display(HTML("<div></div>"))

display(HTML("<div class='section-card'></div>"))
display(segmented_control)

for area in prompt_areas:
    display(area)

# ---------------- Init ----------------
def reset_prompts():
    global prompt_data
    prompt_data = ["", "", "", "", ""]
    for i, area in enumerate(prompt_areas):
        area.value = ""
    display(Javascript("""
        window.promptData = Array(5).fill("");
        if (window.switchPrompt) window.switchPrompt(0);
        const out = document.getElementById('promptOutput');
        if (out) out.textContent = "Generate prompts to see content here...";
    """))

reset_prompts()
status_bar.value = "<div id='statusBar' class='status-text'>Ready — paste script and generate</div>"


HTML(value="<div style='font-size:11px; letter-spacing:1.5px; text-transform:uppercase; font-weight:700; color…

Textarea(value='', layout=Layout(height='240px', width='100%'), placeholder='Paste reference script here...\n(…

HTML(value="<div style='font-size:11px; letter-spacing:1.5px; text-transform:uppercase; font-weight:700; color…

HTML(value="<div style='height:8px'></div>")

HTML(value="<div id='statusBar' class='status-text'>Ready — paste script and generate</div>")

...

Textarea(value='', layout=Layout(height='0px', visibility='hidden'))

Textarea(value='', layout=Layout(height='0px', visibility='hidden'))

Textarea(value='', layout=Layout(height='0px', visibility='hidden'))

Textarea(value='', layout=Layout(height='0px', visibility='hidden'))

<IPython.core.display.Javascript object>

In [112]:
# @title
# JSON → Prompts & SRT (Luxe Purple/Teal UI)
import json, html, threading
import ipywidgets as widgets
from IPython.display import display, Javascript, HTML
from google.colab import files

# ------------------ Theme ------------------
display(HTML(r"""
<style>
@import url('https://fonts.googleapis.com/css2?family=Inter:wght@400;500;600;700;800&display=swap');
:root {
  --bg: #0d0a14; --panel: #18142c; --panel2: #1f1a35;
  --accent: #b277ff; --accent2: #d9a7ff; --teal1: #6ff0e5; --teal2: #7bc5ff;
  --text: #f5f3ff; --muted: #b6a8d9; --radius: 18px;
}
body, .jp-Notebook, .jp-Cell, .jp-OutputArea-output {
  background: radial-gradient(120% 90% at 20% 0%, #1f1534 0%, #0d0a14 45%, #0a0811 100%) !important;
  color: var(--text) !important;
  font-family: 'Inter', system-ui, sans-serif !important;
}
.card {
  background: linear-gradient(145deg, #18142c, #1f1a35);
  border: 1px solid rgba(178,119,255,0.25);
  border-radius: var(--radius);
  padding: 18px;
  box-shadow: 0 18px 50px -14px rgba(0,0,0,0.65);
  margin-bottom: 14px;
}
.hero {
  background: linear-gradient(135deg, #1c1531, #120f22);
  border: 1px solid rgba(178,119,255,0.35);
  border-radius: 20px;
  padding: 22px;
  margin-bottom: 14px;
  box-shadow: 0 24px 60px -18px rgba(0,0,0,0.65);
}
.title { font-size: 22px; font-weight: 800; margin: 0; letter-spacing: -0.3px; }
.subtitle { color: var(--muted); margin: 6px 0 0 0; font-weight: 600; }

/* Primary generate (teal) — unchanged */
.btn {
  border: none; border-radius: 14px; cursor: pointer; font-weight: 800; letter-spacing: 0.2px;
  color: #0c0f16; padding: 14px 22px; height: 50px; display: inline-flex; align-items: center; gap: 8px;
  box-shadow: 0 15px 32px rgba(111,240,229,0.28), inset 0 1px 0 rgba(255,255,255,0.28);
  background: linear-gradient(135deg, var(--teal1), var(--teal2));
}
.btn:hover { transform: translateY(-1px); box-shadow: 0 18px 34px rgba(111,240,229,0.4); }

/* Lavender pill buttons (download + copy) */
.btn-red-sm {
  border: 1px solid rgba(255,255,255,0.18);
  border-radius: 999px;
  cursor: pointer;
  font-weight: 800;
  letter-spacing: 0.15px;
  color: #1b0f34;
  padding: 14px 22px;
  height: 48px;
  display: inline-flex;
  align-items: center;
  justify-content: center;
  gap: 8px;
  line-height: 1.02;
  background: linear-gradient(135deg, #d4b5ff 0%, #c192ff 45%, #a66dff 100%);
  box-shadow: 0 14px 30px rgba(161,109,255,0.30), inset 0 1px 0 rgba(255,255,255,0.20);
  transition: transform 120ms ease, box-shadow 120ms ease, filter 120ms ease, background 160ms ease;
}
.btn-red-sm:hover {
  transform: translateY(-1px);
  box-shadow: 0 18px 36px rgba(161,109,255,0.42);
  filter: brightness(1.05);
}

/* Copy success state — same size, only color changes */
.btn-red-sm.copied {
  background: linear-gradient(135deg, #7ef3cf 0%, #48d7b2 100%);
  color: #063024;
  box-shadow: 0 14px 30px rgba(72,215,178,0.30), inset 0 1px 0 rgba(255,255,255,0.16);
}

/* Textareas */
textarea.jp-RenderedHTMLCommon, textarea {
  background: #120f22 !important;
  border: 1px solid rgba(178,119,255,0.22) !important;
  color: var(--text) !important;
  border-radius: 14px !important;
  font-family: ui-monospace, monospace !important;
}

/* Status */
.status {
  background: rgba(178,119,255,0.10);
  border: 1px solid rgba(178,119,255,0.28);
  color: var(--accent2);
  padding: 10px 14px;
  border-radius: 12px;
  font-weight: 600;
  margin-top: 8px;
}

/* Outputs */
.output-box {
  width: 100%; height: 320px; overflow-y: auto;
  border: 1px solid rgba(178,119,255,0.25);
  padding: 12px;
  background: #120f22;
  color: #e9e5ff;
  font-family: ui-monospace, monospace;
  white-space: pre-wrap;
  line-height: 1.35;
  border-radius: 12px;
}

/* Labels */
.label { font-size: 11px; letter-spacing: 1.2px; text-transform: uppercase; font-weight: 700; color: var(--muted); margin: 6px 0 6px; }
</style>
"""))

# ------------------ Widgets ------------------
json_input = widgets.Textarea(
    placeholder='Paste your JSON here...',
    layout=widgets.Layout(width='100%', height='200px')
)
srt_output = widgets.Textarea(
    placeholder='SRT will appear here...',
    layout=widgets.Layout(width='100%', height='260px')
)
prompts_output = widgets.HTML(value="")
status = widgets.HTML(value="")

generate_btn = widgets.Button(description="✦ GENERATE")
download_srt_btn = widgets.Button(description="DOWNLOAD SRT")
copy_prompts_btn = widgets.Button(description="COPY PROMPTS")

# Classes
generate_btn.add_class("btn")          # keep teal
download_srt_btn.add_class("btn-red-sm")
copy_prompts_btn.add_class("btn-red-sm")

prompts_text_store = ""

# Constraint (unchanged)
constraint = """one image only; landscape or portrait composition; perfectly level; straight-on framing (no tilt/warp/keystone); multi-generation archival reproduction anchor: rephotographed old printed photo / microfilm photocopy / worn album print (not a modern digital photo); no readable text/logos/icons/overlays/watermarks; if any writing exists it must be completely illegible smudges (zero readable characters); ONLY micro-thin beige side strips left/right; no top/bottom border; strips contain no image content; no blur-fill; no mirrored sides; no duplicated/extended background; very faded warm sepia with FADE LOCK: washed-out, very low contrast, matte; lifted blacks; grayish highlights (no bright whites); absolutely no deep blacks; patchy sun-bleach; uneven exposure; milky chemical fog haze; flat tonal range (NOT dark); coarse halftone dot screen across entire photo area + strong dot gain/ink bleed (printed look, not blurry); irregular big clumpy grain (large clumps) + paper fiber texture; heavy damage stack: dense dust/dirt flecks, many long scratches/scuffs, stains/blotches; water marks; grime patches; foxing; emulsion cracking/crazing; banding/posterization; mild print smear/ink spread; repro/photocopy loss artifacts WITHOUT heavy blur; slight ghosting/double-impression; roller streaks; chemical fogging; ringing/halation; subtle macroblocking/mosquito noise; muddy micro-detail; minimal blur only: slight softness at most; motion blur extremely subtle and MUST NOT hit faces; FACE CLARITY LOCK: faces readable (structure visible—eyes/nose/mouth placement), not smeared, not blobbed, not melted; anti-AI look: no HDR, no modern sharpness, no clean gradients, no glossy skin, no cinematic grading, no bokeh, no perfect symmetry; subtly eerie documentary mood (no gore); prefer a posed/held-still moment so faces remain readable"""

# ------------------ Logic ------------------
def generate(b):
    global prompts_text_store
    # reset copy button state on every generate
    copy_prompts_btn.description = "COPY PROMPTS"
    copy_prompts_btn.remove_class("copied")
    copy_prompts_btn.add_class("btn-red-sm")

    status.value = ""
    srt_output.value = ""
    prompts_output.value = ""
    prompts_text_store = ""
    try:
        data = json.loads(json_input.value)
        prompts = []
        srt_blocks = []
        for i, item in enumerate(data, start=1):
            start = item.get("start_time", "").strip()
            end = item.get("end_time", "").strip()
            idea = item.get("main_idea", "").strip()
            prompt = item.get("image_prompt", "").strip()
            if start and end and idea:
                srt_blocks.append(f"{i}\n{start} --> {end}\n{idea}")
            if prompt:
                prompts.append(f"{prompt}\n{constraint}")
        srt_output.value = "\n\n".join(srt_blocks)
        prompts_text_store = "\n\n".join(prompts)
        prompts_output.value = f'<div class="output-box">{html.escape(prompts_text_store)}</div>'
        status.value = '<div class="status" style="color:#8ef0d8;">Done.</div>'
    except Exception as e:
        status.value = f"<div class='status' style='color:#ff9b9b;'>Error: {html.escape(str(e))}</div>"

def copy_prompts(b):
    global prompts_text_store
    if not prompts_text_store.strip():
        status.value = "<div class='status' style='color:#ff9b9b;'>No prompts to copy.</div>"
        return
    safe_text = prompts_text_store.replace("\\", "\\\\").replace("`", "\\`").replace("${", "\\${")
    display(Javascript(f"navigator.clipboard.writeText(`{safe_text}`);"))
    status.value = "<div class='status' style='color:#8ef0d8;'>Prompts copied.</div>"
    # success flash (same size, only color)
    b.description = "COPIED ✓"
    b.add_class("copied")
    def reset_btn():
        b.description = "COPY PROMPTS"
        b.remove_class("copied")
    threading.Timer(1.8, reset_btn).start()

def download_srt(b):
    if not srt_output.value.strip():
        status.value = "<div class='status' style='color:#ff9b9b;'>No SRT to download.</div>"
        return
    with open("subtitles.srt", "w", encoding="utf-8") as f:
        f.write(srt_output.value)
    files.download("subtitles.srt")
    status.value = "<div class='status' style='color:#8ef0d8;'>SRT download started.</div>"

generate_btn.on_click(generate)
copy_prompts_btn.on_click(copy_prompts)
download_srt_btn.on_click(download_srt)

# ------------------ Layout ------------------
display(HTML("""
<div class="hero">
  <div class="title">JSON → Prompts & SRT Tool</div>
  <div class="subtitle">Paste JSON, generate SRT and image prompts with one click.</div>
</div>
"""))

display(HTML('<div class="card">'))
display(HTML('<div class="label">Paste JSON</div>'))
display(json_input)
display(HTML('</div>'))

display(HTML('<div class="card">'))
display(generate_btn)
display(status)
display(HTML('</div>'))

display(HTML('<div class="card">'))
display(HTML('<div class="label">SRT</div>'))
display(srt_output)
display(download_srt_btn)
display(HTML('</div>'))

display(HTML('<div class="card">'))
display(HTML('<div class="label">Prompts</div>'))
display(copy_prompts_btn)
display(prompts_output)
display(HTML('</div>'))


Textarea(value='', layout=Layout(height='200px', width='100%'), placeholder='Paste your JSON here...')

Button(description='✦ GENERATE', style=ButtonStyle(), _dom_classes=('btn',))

HTML(value='')

Textarea(value='', layout=Layout(height='260px', width='100%'), placeholder='SRT will appear here...')

Button(description='DOWNLOAD SRT', style=ButtonStyle(), _dom_classes=('btn-red-sm',))

Button(description='COPY PROMPTS', style=ButtonStyle(), _dom_classes=('btn-red-sm',))

HTML(value='')